In [ ]:
import numpy
import pandas
import uproot

import matplotlib.pyplot as plt
plt.style.use('../../mystyle.mplstyle')

import sbruceana

PATH_TO_SBRUCE = "/Users/triozzi/Analysis/numine/sbruceana/data/cc1mu0pi/"

#### Final selection

In [ ]:
FILE_CV = "final/CNAF_CV_1muNp0pi_NuMI_wMEC.root"
FILE_CV_TRACKBREAKING = "final/CNAF_CV_1muNp0pi_NuMI_trackBreaking.root"
FILE_OFFBEAM = "final/CNAF_OffBeam_1muNp0pi_NuMI.root"
FILE_DATA = "final/CNAF_Data_1muNp0pi_NuMI.root"

In [ ]:
# MC
df = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedNu"  
)
pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedCos"  
)
pot_cos = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos['cosmic'] = 1
df = pandas.concat(
  (df, df_cos)
)

# MC with track breaking
df_tb = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV_TRACKBREAKING}",
  "events/selectedNu"  
)
pot_tb = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV_TRACKBREAKING}")

df_cos_tb = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV_TRACKBREAKING}",
  "events/selectedCos"  
)
pot_cos_tb = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV_TRACKBREAKING}")

df_cos_tb['cosmic'] = 1
df_tb = pandas.concat(
  (df_tb, df_cos_tb)
)

# data
df_data = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_DATA}",
  "events/selectedData"  
)
pot_data = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_DATA}")
time_data = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE}{FILE_DATA}")

# offbeam
df_offbeam = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}",
  "offbeam/selectedOffbeam"  
)
time_offbeam = sbruceana.utils.get_livetime_offbeam(f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "recoE"
bins = numpy.array([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1., 1.1, 1.2, 1.3, 1.4, 1.5, 1.75, 2, 2.5])
# bins = numpy.array([0.35, 0.5, 0.65, 0.8, 1., 1.5, 2.5])


IS_AREA_NORMALIZED = False
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, color='black', marker='.')

# track breaking
df_offbeam['recoE_cathode'] = df_offbeam['recoE']
ax = sbruceana.plotting.plot_var_with_offbeam(ax, df_tb, bins, 'recoE_cathode', df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot_tb, area_normalized=IS_AREA_NORMALIZED, band=False, color='C0', linewidth=1.5, label='cathode break')

df_offbeam['recoE_zeqzero'] = df_offbeam['recoE']
ax = sbruceana.plotting.plot_var_with_offbeam(ax, df_tb, bins, 'recoE_zeqzero', df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot_tb, area_normalized=IS_AREA_NORMALIZED, band=False, color='C1', linewidth=1.5, label='z=0 break')


# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV]',
  ylabel = f'slices [{tag_count}]',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, axes = plt.subplots(figsize=(4, 4), nrows=2, sharex=True)
# bins = numpy.array([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1., 1.1, 1.2, 1.3, 1.4, 1.5, 1.75, 2, 2.5])
bins = numpy.array([0.3, 0.5, 0.65, 0.8, 1., 1.5, 3])

ax = axes[0]
y, edges, _ = ax.hist(df_tb['recoE'], bins=bins, label='CV')

y_cathode, _, _ = ax.hist(df_tb['recoE_cathode'], bins=bins, histtype='step', linewidth=1.5, color='C1', label='data-driven cathode break')

y_z, _, _ = ax.hist(df_tb['recoE_zeqzero'], bins=bins, histtype='step', linewidth=1.5, linestyle='--', color='C2', label='data-driven z=0 break')

ax.legend()

ax = axes[1]

x = (edges[:-1] - edges[1:]) / 2 + edges[1:]

ax.axhline(1, lw=0.75, c='black')
ax.plot(x, y_cathode/y, marker='.', c='C1', label='cathode break')
ax.plot(x, y_z/y, marker='.', c='C2', ls='--', label='z=0 break')
ax.legend()

plt.show()

In [ ]:
def write_ratio_th1(name, values, edges, filename, mode="recreate"):
    """
    values: bin ratios (len = nbins)
    edges:  bin edges (len = nbins + 1)
    """
    with uproot.recreate(filename) if mode == "recreate" else uproot.update(filename) as f:
        f[name] = (values, edges)


ratio_cathode = y_cathode / y
ratio_z0 = y_z / y

sbruceana.utils.convert_ratio_TH1("cathode_break_ratio", ratio_cathode, edges, "CNAF_1muNp0pi_CathodeBreakSyst.root")
sbruceana.utils.convert_ratio_TH1("z0_break_ratio", ratio_z0, edges, "CNAF_1muNp0pi_Z0BreakSyst.root")

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "recoE"
bins = numpy.array([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1., 1.1, 1.2, 1.3, 1.4, 1.5, 1.75, 2, 2.5])

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, color='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV]',
  ylabel = f'slices [{tag_count}]',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "muonl"
width = 25; bins = numpy.arange(50, 500+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, color='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '$L_{\\mu}$ [cm]',
  ylabel = f'slices [{tag_count}]\n/ {width} cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "muonke"
width = 0.1; bins = numpy.arange(0.25, 2+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '$E_{\\mu}$ [GeV]',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "leadpmom"
width = 0.15; bins = numpy.arange(0.2, 1.5+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'leading proton $p$ [GeV]',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "sleadpmom"
width = 0.15; bins = numpy.arange(0.2, 1.5+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'sub-leading proton $p$ [GeV]',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "muchi2mu"
width = 5; bins = numpy.arange(0., 30+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'muon $\chi^2_{\mu}$',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
68.7+8+10.9

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "muchi2pr"
width = 10; bins = numpy.arange(60, 400+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'muon $\chi^2_p$',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "mutrkscore"
width = 0.05; bins = numpy.arange(0, 1+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'muon track score',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "muendx"
width = 2; bins = numpy.arange(-360, 360+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# track breaking
df_offbeam['endX_cathode'] = df_offbeam['muendx']
ax = sbruceana.plotting.plot_var_with_offbeam(ax, df_tb, bins, 'endX_cathode', df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot_tb, area_normalized=IS_AREA_NORMALIZED, band=True, hatch_style='////', color='black', linewidth=1.5)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'muon end $x$',
  ylabel = f'slices [{tag_count}]\n/ {width} cm',
  # xlim   = (bins[0], bins[-1]),
  # xlim = (-230, -190)
  xlim = (190, 230)
)
# leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, axes = plt.subplots(figsize=(4.25, 3.5), ncols=2, layout='constrained', sharey=True)

ax = axes[0]

var = "muendx"
width = 2; bins = numpy.arange(-360, 360+width, width)

IS_AREA_NORMALIZED = True
# ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_var_with_offbeam(ax, df, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, color='black', linewidth=1.5, label='CV')
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# track breaking
df_offbeam['endX_cathode'] = df_offbeam['muendx']
ax = sbruceana.plotting.plot_var_with_offbeam(ax, df_tb, bins, 'endX_cathode', df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot_tb, area_normalized=IS_AREA_NORMALIZED, band=True, hatch_style='////', color='red', linewidth=1.5)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  ylabel = f'slices [{tag_count}] / {width} cm',
  xlim = (-220, -200)
)
ax.set_xticks([-215, -205])
ax.set_yticks([0.005, 0.01])

# leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

ax = axes[1]

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_var_with_offbeam(ax, df, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, color='black', linewidth=1.5, label='CV')
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# track breaking
df_offbeam['endX_cathode'] = df_offbeam['muendx']
ax = sbruceana.plotting.plot_var_with_offbeam(ax, df_tb, bins, 'endX_cathode', df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot_tb, area_normalized=IS_AREA_NORMALIZED, band=True, hatch_style='////', color='red', linewidth=1.5, label='cathode break')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'muon end $x$',
  xlim = (200, 220)
)
ax.set_xticks([205, 215])
leg = ax.legend(fontsize=9.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "muendy"
width = 20; bins = numpy.arange(-200, 160+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'muon end $y$',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "muendz"
width = 2; bins = numpy.arange(-1000, 1000+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_var_with_offbeam(ax, df, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, color='black', linewidth=1.5, label='CV')
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# track breaking
df_offbeam['endZ_zeqzero'] = df_offbeam['muendz']
ax = sbruceana.plotting.plot_var_with_offbeam(ax, df_tb, bins, 'endZ_zeqzero', df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot_tb, area_normalized=IS_AREA_NORMALIZED, band=True, hatch_style='////', color='red', linewidth=1.5, label='$z=0$ break')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'muon end $z$',
  ylabel = f'slices [{tag_count}] / {width} cm',
  # xlim   = (bins[0], bins[-1]),
  xlim = (-20, 20)
)
ax.set_yticks([0.003, 0.006])
leg = ax.legend(fontsize=9.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "muendz"
width = 2; bins = numpy.arange(-1000, 1000+width, width)

IS_AREA_NORMALIZED = False
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# track breaking
df_offbeam['endZ_zeqzero'] = df_offbeam['muendz']
ax = sbruceana.plotting.plot_var_with_offbeam(ax, df_tb, bins, 'endZ_zeqzero', df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot_tb, area_normalized=IS_AREA_NORMALIZED, band=True, hatch_style='////', color='black', linewidth=1.5)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'muon end $z$',
  ylabel = f'slices [{tag_count}]\n/ {width} cm',
  # xlim   = (bins[0], bins[-1]),
  xlim = (-50, 50)
)
# leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "nprotons"
width = 1; bins = numpy.arange(1, 7+width, width)

IS_AREA_NORMALIZED = False
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'N. protons [#]',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "ptrkscore"
width = 0.025; bins = numpy.arange(0, 1+width, width)

IS_AREA_NORMALIZED = False
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'proton track score',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "pchi2mu"
width = 5; bins = numpy.arange(30, 80+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'proton $\\chi^2_{\mu}$',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "pchi2pr"
width = 5; bins = numpy.arange(0, 100+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'proton $\\chi^2_{p}$',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
# resolution
df['muL_res'] = (df['muonl'] - df['truemuonl']) / df['truemuonl']
df['nuE_res'] = (df['recoE'] - df['trueE']) / df['trueE']

df_data['muL_res'] = numpy.zeros(len(df_data))
df_offbeam['muL_res'] = numpy.zeros(len(df_offbeam))

df_data['nuE_res'] = numpy.zeros(len(df_data))
df_offbeam['nuE_res'] = numpy.zeros(len(df_offbeam))


In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "muL_res"
width = 0.02; bins = numpy.arange(-0.5, 0.5+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
# ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '$L-L_\\text{true} / L_\\text{true}$',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "nuE_res"
width = 0.02; bins = numpy.arange(-1, 1+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1MU0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
# ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, c='black', marker='.')

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '$L-L_\\text{true} / L_\\text{true}$',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_numu_{var}.pdf", dpi=300)